<a href="https://colab.research.google.com/github/Komal5578/DWM/blob/main/4_Naive_Bayes_my_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NAIVE BAYES CLASSIFIER: from scratch, no library (paper-ready)
**Edit only the ✏️ DATA cell** (the table, the class column name, and the new tuple to classify).
The output prints every probability as a fraction + decimal, just like the hand solution.

In [2]:
import pandas as pd
from fractions import Fraction
from collections import Counter

## ✏️ DATA: training table + the tuple to classify

In [3]:
columns = ['Outlook','Temp','Humidity','Windy','Play']
data = [
 ['Sunny',   'Hot',  'High',  'False','No'],
 ['Sunny',   'Hot',  'High',  'True', 'No'],
 ['Overcast','Hot',  'High',  'False','Yes'],
 ['Rainy',   'Mild', 'High',  'False','Yes'],
 ['Rainy',   'Cool', 'Normal','False','Yes'],
 ['Rainy',   'Cool', 'Normal','True', 'No'],
 ['Overcast','Cool', 'Normal','True', 'Yes'],
 ['Sunny',   'Mild', 'High',  'False','No'],
 ['Sunny',   'Cool', 'Normal','False','Yes'],
 ['Rainy',   'Mild', 'Normal','False','Yes'],
 ['Sunny',   'Mild', 'Normal','True', 'Yes'],
 ['Overcast','Mild', 'High',  'True', 'Yes'],
 ['Overcast','Hot',  'Normal','False','Yes'],
 ['Rainy',   'Mild', 'High',  'True', 'No'],
]
CLASS_COL = 'Play'
new_tuple = {'Outlook':'Sunny', 'Temp':'Cool', 'Humidity':'High', 'Windy':'True'}   # X to classify
LAPLACE = False      # set True if a probability becomes 0 (Laplace / add-one smoothing)

df = pd.DataFrame(data, columns=columns)
df

,Outlook,Temp,Humidity,Windy,Play
0,Sunny,Hot,High,False,No
1,Sunny,Hot,High,True,No
2,Overcast,Hot,High,False,Yes
3,Rainy,Mild,High,False,Yes
4,Rainy,Cool,Normal,False,Yes
5,Rainy,Cool,Normal,True,No
6,Overcast,Cool,Normal,True,Yes
7,Sunny,Mild,High,False,No
8,Sunny,Cool,Normal,False,Yes
9,Rainy,Mild,Normal,False,Yes


## Step 1: Prior probabilities  P(class)

In [4]:
n = len(df)
class_count = Counter(df[CLASS_COL])
prior = {c: Fraction(cnt, n) for c, cnt in class_count.items()}
for c in prior: print(f'P({CLASS_COL}={c}) = {class_count[c]}/{n} = {float(prior[c]):.4f}')

P(Play=No) = 5/14 = 0.3571
P(Play=Yes) = 9/14 = 0.6429


## Step 2: Frequency tables (count of each value per class)

In [5]:
attrs = [c for c in df.columns if c != CLASS_COL]
for a in attrs:
    print(pd.crosstab(df[a], df[CLASS_COL]), '\n')

Play      No  Yes
Outlook          
Overcast   0    4
Rainy      2    3
Sunny      3    2 

Play  No  Yes
Temp         
Cool   1    3
Hot    2    2
Mild   2    4 

Play      No  Yes
Humidity         
High       4    3
Normal     1    6 

Play   No  Yes
Windy         
False   2    6
True    3    3 



## Step 3: Conditional probabilities  P(attribute = value | class) for the new tuple

In [6]:
def cond_prob(attr, value, c):
    num = ((df[attr] == value) & (df[CLASS_COL] == c)).sum()
    den = class_count[c]
    if LAPLACE:
        num += 1
        den += df[attr].nunique()
    return Fraction(int(num), int(den))

scores = {}
for c in prior:
    score = prior[c]
    print(f'--- class = {c} ---')
    for a in attrs:
        p = cond_prob(a, new_tuple[a], c)
        print(f'  P({a}={new_tuple[a]} | {c}) = {p} = {float(p):.4f}')
        score *= p
    scores[c] = score
    print(f'  P(X|{c}) x P({c}) = {float(score):.6f}\n')

--- class = No ---
  P(Outlook=Sunny | No) = 3/5 = 0.6000
  P(Temp=Cool | No) = 1/5 = 0.2000
  P(Humidity=High | No) = 4/5 = 0.8000
  P(Windy=True | No) = 3/5 = 0.6000
  P(X|No) x P(No) = 0.020571

--- class = Yes ---
  P(Outlook=Sunny | Yes) = 2/9 = 0.2222
  P(Temp=Cool | Yes) = 1/3 = 0.3333
  P(Humidity=High | Yes) = 1/3 = 0.3333
  P(Windy=True | Yes) = 1/3 = 0.3333
  P(X|Yes) x P(Yes) = 0.005291



## Step 4: Decision (pick the class with the largest score)

In [7]:
total = sum(scores.values())
for c, s in scores.items():
    print(f'Score({c}) = {float(s):.6f}   normalised probability = {float(s/total):.4f}' if total else f'Score({c}) = 0')
print('\nPREDICTED CLASS  ->', max(scores, key=scores.get))

Score(No) = 0.020571   normalised probability = 0.7954
Score(Yes) = 0.005291   normalised probability = 0.2046

PREDICTED CLASS  -> No


## (Optional) Accuracy of the model on the training table

In [8]:
def predict(row):
    sc = {}
    for c in prior:
        s = prior[c]
        for a in attrs: s *= cond_prob(a, row[a], c)
        sc[c] = s
    return max(sc, key=sc.get)

df['Predicted'] = df.apply(predict, axis=1)
print('Accuracy:', (df['Predicted'] == df[CLASS_COL]).mean().round(3))
print(pd.crosstab(df[CLASS_COL], df['Predicted'], rownames=['Actual'], colnames=['Predicted']))

Accuracy: 0.929
Predicted  No  Yes
Actual            
No          4    1
Yes         0    9


**Formula to write:** P(C|X) ∝ P(C) · ∏ P(xᵢ|C)  (assumes attributes are independent given the class → "naive").
**Zero probability problem:** one zero kills the product → use Laplace smoothing (`LAPLACE = True`: add 1 to each count, add number of distinct values to the denominator).